In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install deepface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.9 MB/s eta 0:00:00


In [4]:
import os
import cv2
import numpy as np
import pickle
from deepface import DeepFace

# Đường dẫn trên Google Drive của bạn
DATASET_PATH = "/content/drive/MyDrive/khuong_mat/dataset"
DB_PATH = "/content/drive/MyDrive/khuong_mat/face_db.pkl"

def process_dataset(dataset_path, db_path):
    face_database = {}

    if not os.path.exists(dataset_path):
        print(f"LỖI: Không tìm thấy thư mục dataset tại {dataset_path}. Hãy kiểm tra lại cấu trúc thư mục trên Drive.")
        return

    # Lấy danh sách các thư mục con (tên từng người, VD: "Bang")
    people_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]

    if not people_folders:
        print("Không tìm thấy thư mục con nào trong dataset.")
        return

    for person_name in people_folders:
        print(f"Đang xử lý dữ liệu cho: {person_name}...")
        face_database[person_name] = []

        person_dir = os.path.join(dataset_path, person_name)
        images_dir = os.path.join(person_dir, "images")
        labels_dir = os.path.join(person_dir, "labels")

        if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
            print(f" -> Bỏ qua {person_name} vì thiếu thư mục 'images' hoặc 'labels'.")
            continue

        # Duyệt qua từng ảnh trong thư mục images
        for img_name in os.listdir(images_dir):
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            img_path = os.path.join(images_dir, img_name)
            label_name = os.path.splitext(img_name)[0] + ".txt"
            label_path = os.path.join(labels_dir, label_name)

            if not os.path.exists(label_path):
                continue

            # Đọc ảnh
            img = cv2.imread(img_path)
            if img is None:
                continue
            h, w, _ = img.shape

            # Đọc tọa độ từ file txt (Format YOLO)
            with open(label_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    # Chuyển đổi tọa độ YOLO sang pixel
                    center_x = float(parts[1]) * w
                    center_y = float(parts[2]) * h
                    box_w = float(parts[3]) * w
                    box_h = float(parts[4]) * h

                    x1 = int(center_x - box_w / 2)
                    y1 = int(center_y - box_h / 2)
                    x2 = int(center_x + box_w / 2)
                    y2 = int(center_y + box_h / 2)

                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(w, x2), min(h, y2)

                    # Cắt khuôn mặt
                    face_crop = img[y1:y2, x1:x2]

                    if face_crop.shape[0] < 20 or face_crop.shape[1] < 20:
                        continue

                    try:
                        # Trích xuất vector embedding bằng Facenet512
                        embedding_objs = DeepFace.represent(
                            img_path=face_crop,
                            model_name="Facenet512",
                            enforce_detection=False
                        )
                        embedding = embedding_objs[0]["embedding"]
                        face_database[person_name].append(embedding)

                    except Exception as e:
                        print(f"Lỗi trích xuất file {img_name}: {e}")

        print(f" -> Hoàn thành: Trích xuất được {len(face_database[person_name])} vector cho {person_name}.\n")

    # Lưu file database trực tiếp vào Google Drive để không bị mất khi tắt Colab
    with open(db_path, "wb") as f:
        pickle.dump(face_database, f)
    print(f"THÀNH CÔNG: Đã lưu Database vector tại: {db_path}")

# Kích hoạt hàm xử lý
process_dataset(DATASET_PATH, DB_PATH)

26-06-06 15:09:24 - Directory /root/.deepface has been created
26-06-06 15:09:24 - Directory /root/.deepface/weights has been created
Đang xử lý dữ liệu cho: Bang...
26-06-06 15:09:33 - 🔗 facenet512_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.h5 to /root/.deepface/weights/facenet512_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.h5
To: /root/.deepface/weights/facenet512_weights.h5
100%|██████████| 95.0M/95.0M [00:02<00:00, 39.4MB/s]


 -> Hoàn thành: Trích xuất được 9 vector cho Bang.

Đang xử lý dữ liệu cho: lap...
 -> Hoàn thành: Trích xuất được 7 vector cho lap.

THÀNH CÔNG: Đã lưu Database vector tại: /content/drive/MyDrive/khuong_mat/face_db.pkl
